In [1]:
import os
from langchain_ollama import OllamaLLM
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain.prompts import PromptTemplate

# 1. Ollama를 통한 Gemma 설정
# 사용자의 환경에 맞춰 모델명과 base_url을 유지합니다.
llm = OllamaLLM(
    model="gemma4:26b", 
    temperature=0,
    base_url="http://172.27.48.1:11434", 
    timeout=900,
    model_kwargs={
        "num_gpu": 99,       
        "main_gpu": 0,
    }
)

# 2. 도구(Tools) 정의
def read_file(path):
    try:
        with open(path.strip(), 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e: return f"에러: {e}"

def write_file(input_str):
    try:
        # '파일명||내용' 형식을 체크합니다.
        if "||" not in input_str:
             return "에러: '파일명||내용' 형식을 지켜주세요. 예: test.txt||hello world"
        path, content = input_str.split("||", 1)
        with open(path.strip(), 'w', encoding='utf-8') as f:
            f.write(content)
        return f"{path} 저장 완료"
    except Exception as e: return f"에러: {e}"

tools = [
    Tool(name="ReadFile", func=read_file, description="파일 내용을 읽습니다. 인자: 파일경로"),
    Tool(name="WriteFile", func=write_file, description="파일을 씁니다. 인자: '파일명||내용'")
]

# 3. 에이전트 생성 (ReAct 프롬프트 강화)
# Gemma가 'Action:' 태그를 누락하지 않도록 지시사항을 명확히 한 커스텀 프롬프트를 사용합니다.
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

CRITICAL RULE: 
Every 'Thought:' MUST be immediately followed by either an 'Action:' or a 'Final Answer'. 
Never stop after a 'Thought:' without providing an 'Action:' or the 'Final Answer'.

Begin!

모든 답변은 반드시 'Thought:'로 시작하며, 그 뒤에는 도구를 사용하기 위한 'Action:' 또는 최종 답변인 'Final Answer:'가 반드시 이어져야 합니다.

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors="Invalid Format: Missing 'Action:' after 'Thought:'. Please follow the format: Thought: [your thought] then Action: [tool name] then Action Input: [input]"
)

print("✅ Gemma Agent 2 준비 완료! (프롬프트 강화 및 파싱 에러 처리 개선)")

✅ Gemma Agent 2 준비 완료! (프롬프트 강화 및 파싱 에러 처리 개선)


In [2]:
# 실행 테스트
agent_executor.invoke({"input": 
"""
로컬환경에 있는 gemma4:26b 모델을 활용해서 RAG와 Langchain, LangGraph를 적용하고 싶어.

국가 R&D 부적정 사례집.pdf
국가연구개발사업 연구개발비 사용 기준.pdf
국가연구개발혁신법.pdf

위의 세 가지 파일을 RAG 학습하고, python 파일을 실행하면 사용자가 입력한 값을 테스트 하는 파일을 만들어줘
python 파일의 이름은 gemma_rag.py 로 만들어주고, requirements.txt 파일도 만들어줘
README.md 파일도 만들어줘
"""
})



> Entering new AgentExecutor chain...
Thought:
사용자의 요구사항에 따라 `gemma4:26b` 모델을 활용한 RAG 시스템을 구축하기 위한 코드를 작성하겠습니다. 이 시스템은 LangChain과 LangGraph를 사용하며, 제공된 세 개의 PDF 파일을 기반으로 답변을 생성합니다.

먼저, 프로젝트에 필요한 라이브기들을 정의하는 `requirements.txt` 파일을 생성하겠습니다.

Action: WriteFile
Action Input: requirements.txt||langchain
langchain-community
langchain-ollama
langgraph
pypdf
chromadb
sentence-transformers
python-dotenv

<channel|>Thought:
다음으로, 핵심 로직이 담긴 `gemma_rag.py` 파일을 작성하겠습니다. 이 파일은 PDF 로드, 텍스트 분할, 임베딩 생성, 벡터 스토어 저장, 그리고 LangGraph를 이용한 RAG 워크플로우를 포함합니다. 모델은 로컬에 설치된 `gemma4:26b` (Ollama 기준)를 사용한다고 가정합니다.

Action: WriteFile
Action Input: gemma_rag.py||import os
from typing import Annotated, List, TypedDict
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage, Sy

{'input': '\n로컬환경에 있는 gemma4:26b 모델을 활용해서 RAG와 Langchain, LangGraph를 적용하고 싶어.\n\n국가 R&D 부적정 사례집.pdf\n국가연구개발사업 연구개발비 사용 기준.pdf\n국가연구개발혁신법.pdf\n\n위의 세 가지 파일을 RAG 학습하고, python 파일을 실행하면 사용자가 입력한 값을 테스트 하는 파일을 만들어줘\npython 파일의 이름은 gemma_rag.py 로 만들어주고, requirements.txt 파일도 만들어줘\nREADME.md 파일도 만들어줘\n',
 'output': 'Agent stopped due to iteration limit or time limit.'}